In [2]:
import glob
import pandas as pd

# 1. 12개 CSV 파일 불러오기 및 합치기
# (파일명 패턴에 맞게 경로 수정 필요, 한글 깨짐 발생 시 encoding='utf-8'로 변경)
file_paths = sorted(glob.glob('data/*2025*.csv'))
target_columns = ["대여일자", "대여시간", "대여소번호", "대여소명", "이용건수"]

df_list = []
for file in file_paths:
    df = pd.read_csv(file, encoding="cp949",usecols=target_columns, engine="c") 
    df_list.append(df)

df_bike = pd.concat(df_list, ignore_index=True)

In [3]:
df_bike.head()

,대여일자,대여시간,대여소번호,대여소명,이용건수
0,2025-01-01,0,3684,3684. 고덕자이(105동 앞),1
1,2025-01-01,0,1044,1044. 굽은다리역,1
2,2025-01-01,0,1153,"1153. 발산역 1번, 9번 인근 대여소",1
3,2025-01-01,0,1260,1260. 방이동 한양3차아파트 옆,1
4,2025-01-01,0,1845,1845. 롯데캐슬골드파크1차 서문,1


In [4]:
# 2. 날짜+시간 컬럼(datetime) 생성
# '대여일자'(2025-01-01) + '대여시간'(0~23)을 datetime 객체로 변환
df_bike["datetime"] = pd.to_datetime(
    df_bike["대여일자"]
) + pd.to_timedelta(df_bike["대여시간"], unit="h")

# 3. [시간대 + 대여소] 기준 이용건수 합계 계산 (성별/연령대별 행 통합)
df_daily_station = (
    df_bike.groupby(["datetime", "대여일자", "대여시간", "대여소번호","대여소명"])[
        "이용건수"
    ]
    .sum()
    .reset_index()
)

In [5]:
df_daily_station.head(10)

,datetime,대여일자,대여시간,대여소번호,대여소명,이용건수
0,2025-01-01,2025-01-01,0,102,102. 망원역 1번출구 앞,2
1,2025-01-01,2025-01-01,0,103,103. 망원역 2번출구 앞,1
2,2025-01-01,2025-01-01,0,104,104. 합정역 1번출구 앞,1
3,2025-01-01,2025-01-01,0,105,105. 합정역 5번출구 앞,3
4,2025-01-01,2025-01-01,0,107,107. 신한은행 서교동지점,1
5,2025-01-01,2025-01-01,0,108,108. 서교동 사거리,1
6,2025-01-01,2025-01-01,0,111,111. 상수역 2번출구 앞,1
7,2025-01-01,2025-01-01,0,113,113. 홍대입구역 2번출구 앞,3
8,2025-01-01,2025-01-01,0,114,114. 홍대입구역 8번출구 앞,2
9,2025-01-01,2025-01-01,0,120,120. 신수동 사거리,1


In [6]:
df_weather = pd.read_csv('data/2025년도 날씨데이터.csv', encoding="cp949")
df_weather

,지점,지점명,일시,기온(°C),강수량(mm),풍속(m/s),습도(%)
0,108,서울,2025-01-01 00:00,-1.2,NaN,0.7,60
1,108,서울,2025-01-01 01:00,-1.7,NaN,1.1,62
2,108,서울,2025-01-01 02:00,-1.8,NaN,0.5,64
3,108,서울,2025-01-01 03:00,-2.0,NaN,1.9,66
4,108,서울,2025-01-01 04:00,-2.3,NaN,2.2,67
...,...,...,...,...,...,...,...
8755,108,서울,2025-12-31 19:00,-5.6,NaN,3.7,41
8756,108,서울,2025-12-31 20:00,-6.3,NaN,3.5,45
8757,108,서울,2025-12-31 21:00,-7.4,NaN,2.2,48
8758,108,서울,2025-12-31 22:00,-8.1,NaN,1.4,47


In [8]:
# 기상청 데이터의 '일시' 컬럼(예: '2025-01-01 00:00')을 datetime으로 변환
df_weather["datetime"] = pd.to_datetime(df_weather["일시"])

# 필요한 기상 요소만 선택 (예: 기온, 강수량, 풍속, 습도 등)
weather_cols = [
    "datetime",
    "기온(°C)",
    "강수량(mm)",
    "풍속(m/s)",
    "습도(%)",
]
df_weather_sub = df_weather[weather_cols]

In [9]:
# 5. 자전거 데이터와 기상 데이터 Merge (시간 기준 Left Join)
df_final = pd.merge(
    df_daily_station, df_weather_sub, on="datetime", how="left"
)

In [10]:
df_final.head(10)

,datetime,대여일자,대여시간,대여소번호,대여소명,이용건수,기온(°C),강수량(mm),풍속(m/s),습도(%)
0,2025-01-01,2025-01-01,0,102,102. 망원역 1번출구 앞,2,-1.2,NaN,0.7,60
1,2025-01-01,2025-01-01,0,103,103. 망원역 2번출구 앞,1,-1.2,NaN,0.7,60
2,2025-01-01,2025-01-01,0,104,104. 합정역 1번출구 앞,1,-1.2,NaN,0.7,60
3,2025-01-01,2025-01-01,0,105,105. 합정역 5번출구 앞,3,-1.2,NaN,0.7,60
4,2025-01-01,2025-01-01,0,107,107. 신한은행 서교동지점,1,-1.2,NaN,0.7,60
5,2025-01-01,2025-01-01,0,108,108. 서교동 사거리,1,-1.2,NaN,0.7,60
6,2025-01-01,2025-01-01,0,111,111. 상수역 2번출구 앞,1,-1.2,NaN,0.7,60
7,2025-01-01,2025-01-01,0,113,113. 홍대입구역 2번출구 앞,3,-1.2,NaN,0.7,60
8,2025-01-01,2025-01-01,0,114,114. 홍대입구역 8번출구 앞,2,-1.2,NaN,0.7,60
9,2025-01-01,2025-01-01,0,120,120. 신수동 사거리,1,-1.2,NaN,0.7,60


In [11]:
# 6. 결측치 처리 (비가 안 온 날 강수량이 NaN인 경우 0으로 채우기)
df_final["강수량(mm)"] = df_final["강수량(mm)"].fillna(0)

# 최종 확인 및 저장
df_final.head()

,datetime,대여일자,대여시간,대여소번호,대여소명,이용건수,기온(°C),강수량(mm),풍속(m/s),습도(%)
0,2025-01-01,2025-01-01,0,102,102. 망원역 1번출구 앞,2,-1.2,0.0,0.7,60
1,2025-01-01,2025-01-01,0,103,103. 망원역 2번출구 앞,1,-1.2,0.0,0.7,60
2,2025-01-01,2025-01-01,0,104,104. 합정역 1번출구 앞,1,-1.2,0.0,0.7,60
3,2025-01-01,2025-01-01,0,105,105. 합정역 5번출구 앞,3,-1.2,0.0,0.7,60
4,2025-01-01,2025-01-01,0,107,107. 신한은행 서교동지점,1,-1.2,0.0,0.7,60


In [14]:
df_final = df_final.rename(columns={'datetime':'대여일시'})
df_final.head(10)

,대여일시,대여일자,대여시간,대여소번호,대여소명,이용건수,기온(°C),강수량(mm),풍속(m/s),습도(%)
0,2025-01-01,2025-01-01,0,102,102. 망원역 1번출구 앞,2,-1.2,0.0,0.7,60
1,2025-01-01,2025-01-01,0,103,103. 망원역 2번출구 앞,1,-1.2,0.0,0.7,60
2,2025-01-01,2025-01-01,0,104,104. 합정역 1번출구 앞,1,-1.2,0.0,0.7,60
3,2025-01-01,2025-01-01,0,105,105. 합정역 5번출구 앞,3,-1.2,0.0,0.7,60
4,2025-01-01,2025-01-01,0,107,107. 신한은행 서교동지점,1,-1.2,0.0,0.7,60
5,2025-01-01,2025-01-01,0,108,108. 서교동 사거리,1,-1.2,0.0,0.7,60
6,2025-01-01,2025-01-01,0,111,111. 상수역 2번출구 앞,1,-1.2,0.0,0.7,60
7,2025-01-01,2025-01-01,0,113,113. 홍대입구역 2번출구 앞,3,-1.2,0.0,0.7,60
8,2025-01-01,2025-01-01,0,114,114. 홍대입구역 8번출구 앞,2,-1.2,0.0,0.7,60
9,2025-01-01,2025-01-01,0,120,120. 신수동 사거리,1,-1.2,0.0,0.7,60


In [16]:
df_final = df_final.drop(["대여일자", "대여시간"], axis=1)

In [17]:
df_final.head(10)

,대여일시,대여소번호,대여소명,이용건수,기온(°C),강수량(mm),풍속(m/s),습도(%)
0,2025-01-01,102,102. 망원역 1번출구 앞,2,-1.2,0.0,0.7,60
1,2025-01-01,103,103. 망원역 2번출구 앞,1,-1.2,0.0,0.7,60
2,2025-01-01,104,104. 합정역 1번출구 앞,1,-1.2,0.0,0.7,60
3,2025-01-01,105,105. 합정역 5번출구 앞,3,-1.2,0.0,0.7,60
4,2025-01-01,107,107. 신한은행 서교동지점,1,-1.2,0.0,0.7,60
5,2025-01-01,108,108. 서교동 사거리,1,-1.2,0.0,0.7,60
6,2025-01-01,111,111. 상수역 2번출구 앞,1,-1.2,0.0,0.7,60
7,2025-01-01,113,113. 홍대입구역 2번출구 앞,3,-1.2,0.0,0.7,60
8,2025-01-01,114,114. 홍대입구역 8번출구 앞,2,-1.2,0.0,0.7,60
9,2025-01-01,120,120. 신수동 사거리,1,-1.2,0.0,0.7,60


In [19]:
df_final['대여소명'].head(10)

0      102. 망원역 1번출구 앞
1      103. 망원역 2번출구 앞
2      104. 합정역 1번출구 앞
3      105. 합정역 5번출구 앞
4      107. 신한은행 서교동지점
5         108. 서교동 사거리
6      111. 상수역 2번출구 앞
7    113. 홍대입구역 2번출구 앞
8    114. 홍대입구역 8번출구 앞
9         120. 신수동 사거리
Name: 대여소명, dtype: str

In [ ]:
df_final['대여소명'] = df_final['대여소명'].str.replace(r'^[0-9]+\.\s*', '', regex=True)

In [21]:
df_final.head(10)

,대여일시,대여소번호,대여소명,이용건수,기온(°C),강수량(mm),풍속(m/s),습도(%)
0,2025-01-01,102,망원역 1번출구 앞,2,-1.2,0.0,0.7,60
1,2025-01-01,103,망원역 2번출구 앞,1,-1.2,0.0,0.7,60
2,2025-01-01,104,합정역 1번출구 앞,1,-1.2,0.0,0.7,60
3,2025-01-01,105,합정역 5번출구 앞,3,-1.2,0.0,0.7,60
4,2025-01-01,107,신한은행 서교동지점,1,-1.2,0.0,0.7,60
5,2025-01-01,108,서교동 사거리,1,-1.2,0.0,0.7,60
6,2025-01-01,111,상수역 2번출구 앞,1,-1.2,0.0,0.7,60
7,2025-01-01,113,홍대입구역 2번출구 앞,3,-1.2,0.0,0.7,60
8,2025-01-01,114,홍대입구역 8번출구 앞,2,-1.2,0.0,0.7,60
9,2025-01-01,120,신수동 사거리,1,-1.2,0.0,0.7,60


In [31]:
# 1. 상단 헤더 3줄을 모두 컬럼 레이어로 가져오기 (실제 줄 수에 맞춰 지정)
df_bike_info = pd.read_excel(
    'data/공공자전거 대여소 정보(25.12월 기준).xlsx',
    header=[0, 1, 2]
)

# 2. 다중 헤더를 언더바(_)로 연결하여 하나의 컬럼명으로 압축하기
# 예: ('소재지(위치)', '상세주소') -> '소재지(위치)_상세주소'
new_columns = []
for col in df_bike_info.columns:
    # Unnamed가 포함된 상위 레이어 이름은 버리고 의미 있는 이름만 조합
    cleaned_col = [c for c in col if "Unnamed" not in c]
    new_columns.append("_".join(cleaned_col))

df_bike_info.columns = new_columns

# 3. 앞뒤 공백 정리
df_bike_info.columns = df_bike_info.columns.str.strip()

In [40]:
# 1. '거치대수' 항목을 상단 컬럼명에 합쳐서 직접 지정하기
df_bike_info = df_bike_info.rename(
    columns={
        "설치형태_LCD": "설치형태_LCD_거치대수",
        "설치형태_QR": "설치형태_QR_거치대수",
    }
)

# 2. 데이터가 없는 상위 0번, 1번 행(Index 0, 1) 제거하기
df_bike_info = df_bike_info.drop([0, 1], axis=0)

# 3. 컬럼 이름에 들어있는 줄바꿈('\n') 문자 깔끔하게 제거하기
# 예: '대여소\n번호' -> '대여소번호', '설치\n시기' -> '설치시기'
df_bike_info.columns = df_bike_info.columns.str.replace("\n", "")

# 4. (선택) 행을 지웠으므로 인덱스 번호를 0부터 다시 예쁘게 정렬하기
df_bike_info = df_bike_info.reset_index(drop=True)

# 결과 확인
df_bike_info.head(10)

,대여소번호,보관소(대여소)명,소재지(위치)_자치구,소재지(위치)_상세주소,소재지(위치)_위도,소재지(위치)_경도,설치시기,설치형태_LCD_거치대수,설치형태_QR_거치대수,운영방식
0,102.0,망원역 1번출구 앞,마포구,서울특별시 마포구 월드컵로 72,37.555649,126.910629,2015-09-06 23:42:06,NaN,15,QR
1,103.0,망원역 2번출구 앞,마포구,서울특별시 마포구 월드컵로 79,37.554951,126.910835,2015-09-06 23:43:13,NaN,14,QR
2,104.0,합정역 1번출구 앞,마포구,서울특별시 마포구 양화로 59,37.550629,126.914986,2015-09-06 23:44:31,NaN,13,QR
3,105.0,합정역 5번출구 앞,마포구,서울특별시 마포구 양화로 48,37.550007,126.914825,2015-09-06 23:45:30,NaN,5,QR
4,106.0,합정역 7번출구 앞,마포구,서울특별시 마포구 독막로 4,37.548645,126.912827,2015-09-06 23:46:31,NaN,12,QR
5,107.0,신한은행 서교동지점,마포구,서울특별시 마포구 월드컵북로 35,37.557510,126.918503,2015-09-06 23:47:57,NaN,5,QR
6,108.0,서교동 사거리,마포구,서울특별시 마포구 양화로 93,37.552746,126.918617,2015-09-06 23:51:25,NaN,10,QR
7,111.0,상수역 2번출구 앞,마포구,서울특별시 마포구 와우산로 40,37.547871,126.923531,2015-09-07 01:30:37,NaN,10,QR
8,113.0,홍대입구역 2번출구 앞,마포구,서울특별시 마포구 양화로 165,37.557438,126.923821,2015-09-07 01:32:58,NaN,25,QR
9,114.0,홍대입구역 8번출구 앞,마포구,서울특별시 마포구 양화로18길 3,37.557060,126.924423,2015-09-07 01:33:52,NaN,15,QR


In [43]:
df_bike_info = df_bike_info.drop(['운영방식'], axis=1)

In [44]:
df_bike_info.head(10)

,대여소번호,보관소(대여소)명,소재지(위치)_자치구,소재지(위치)_상세주소,설치형태_LCD_거치대수,설치형태_QR_거치대수
0,102.0,망원역 1번출구 앞,마포구,서울특별시 마포구 월드컵로 72,NaN,15
1,103.0,망원역 2번출구 앞,마포구,서울특별시 마포구 월드컵로 79,NaN,14
2,104.0,합정역 1번출구 앞,마포구,서울특별시 마포구 양화로 59,NaN,13
3,105.0,합정역 5번출구 앞,마포구,서울특별시 마포구 양화로 48,NaN,5
4,106.0,합정역 7번출구 앞,마포구,서울특별시 마포구 독막로 4,NaN,12
5,107.0,신한은행 서교동지점,마포구,서울특별시 마포구 월드컵북로 35,NaN,5
6,108.0,서교동 사거리,마포구,서울특별시 마포구 양화로 93,NaN,10
7,111.0,상수역 2번출구 앞,마포구,서울특별시 마포구 와우산로 40,NaN,10
8,113.0,홍대입구역 2번출구 앞,마포구,서울특별시 마포구 양화로 165,NaN,25
9,114.0,홍대입구역 8번출구 앞,마포구,서울특별시 마포구 양화로18길 3,NaN,15


In [45]:
# 1. LCD와 QR 거치대수를 하나로 합치기
# 두 컬럼 중 값이 있는 것을 가져옵니다. 둘 다 NaN이면 결측치 처리됩니다.
df_bike_info["거치대수"] = df_bike_info["설치형태_QR_거치대수"].fillna(
    df_bike_info["설치형태_LCD_거치대수"]
)

In [46]:
# 2. 합치고 남은 기존 LCD, QR 컬럼은 삭제하기
df_bike_info = df_bike_info.drop(
    ["설치형태_LCD_거치대수", "설치형태_QR_거치대수"], axis=1
)

In [48]:
df_bike_info.head(10)

,대여소번호,보관소(대여소)명,소재지(위치)_자치구,소재지(위치)_상세주소,거치대수
0,102.0,망원역 1번출구 앞,마포구,서울특별시 마포구 월드컵로 72,15
1,103.0,망원역 2번출구 앞,마포구,서울특별시 마포구 월드컵로 79,14
2,104.0,합정역 1번출구 앞,마포구,서울특별시 마포구 양화로 59,13
3,105.0,합정역 5번출구 앞,마포구,서울특별시 마포구 양화로 48,5
4,106.0,합정역 7번출구 앞,마포구,서울특별시 마포구 독막로 4,12
5,107.0,신한은행 서교동지점,마포구,서울특별시 마포구 월드컵북로 35,5
6,108.0,서교동 사거리,마포구,서울특별시 마포구 양화로 93,10
7,111.0,상수역 2번출구 앞,마포구,서울특별시 마포구 와우산로 40,10
8,113.0,홍대입구역 2번출구 앞,마포구,서울특별시 마포구 양화로 165,25
9,114.0,홍대입구역 8번출구 앞,마포구,서울특별시 마포구 양화로18길 3,15


In [49]:
df_bike_info["대여소번호"] = (
    pd.to_numeric(df_bike_info["대여소번호"], errors="coerce")
    .fillna(0)
    .astype(int)
    .astype(str)
)

In [51]:
df_bike_info.head()

,대여소번호,보관소(대여소)명,소재지(위치)_자치구,소재지(위치)_상세주소,거치대수
0,102,망원역 1번출구 앞,마포구,서울특별시 마포구 월드컵로 72,15
1,103,망원역 2번출구 앞,마포구,서울특별시 마포구 월드컵로 79,14
2,104,합정역 1번출구 앞,마포구,서울특별시 마포구 양화로 59,13
3,105,합정역 5번출구 앞,마포구,서울특별시 마포구 양화로 48,5
4,106,합정역 7번출구 앞,마포구,서울특별시 마포구 독막로 4,12


In [ ]:
df_bike_info = df_bike_info.rename(columns={'보관소(대여소)명': '대여소명', '소재지(위치)_자치구': '자치구','소재지(위치)_상세주소':'상세주소'})

In [53]:
df_bike_info.head()

,대여소번호,대여소명,자치구,상세주소,거치대수
0,102,망원역 1번출구 앞,마포구,서울특별시 마포구 월드컵로 72,15
1,103,망원역 2번출구 앞,마포구,서울특별시 마포구 월드컵로 79,14
2,104,합정역 1번출구 앞,마포구,서울특별시 마포구 양화로 59,13
3,105,합정역 5번출구 앞,마포구,서울특별시 마포구 양화로 48,5
4,106,합정역 7번출구 앞,마포구,서울특별시 마포구 독막로 4,12


In [54]:
df_final.head()

,대여일시,대여소번호,대여소명,이용건수,기온(°C),강수량(mm),풍속(m/s),습도(%)
0,2025-01-01,102,망원역 1번출구 앞,2,-1.2,0.0,0.7,60
1,2025-01-01,103,망원역 2번출구 앞,1,-1.2,0.0,0.7,60
2,2025-01-01,104,합정역 1번출구 앞,1,-1.2,0.0,0.7,60
3,2025-01-01,105,합정역 5번출구 앞,3,-1.2,0.0,0.7,60
4,2025-01-01,107,신한은행 서교동지점,1,-1.2,0.0,0.7,60


In [56]:
# 1. 머지 전 대여소번호 데이터 타입 일치시키기 (오류 방지)
# 두 데이터프레임의 '대여소번호' 컬럼 타입이 다르면(예: 문자열 vs 정수) 머지가 안 될 수 있으므로 정수(int)로 맞춥니다.
df_final["대여소번호"] = df_final["대여소번호"].astype(int)
df_bike_info["대여소번호"] = df_bike_info["대여소번호"].astype(int)

# 2. df_bike_info에서 필요한 컬럼만 추출하여 머지
# '대여소명'은 df_final에 이미 있으므로 제외하고 ['대여소번호', '자치구', '상세주소', '거치대수']만 가져옵니다.
info_cols = ["대여소번호", "자치구", "상세주소", "거치대수"]

df_final_merged = pd.merge(
    df_final, df_bike_info[info_cols], on="대여소번호", how="left"
)

# 3. 결과 확인 및 저장
df_final_merged.head()

,대여일시,대여소번호,대여소명,이용건수,기온(°C),강수량(mm),풍속(m/s),습도(%),자치구,상세주소,거치대수
0,2025-01-01,102,망원역 1번출구 앞,2,-1.2,0.0,0.7,60,마포구,서울특별시 마포구 월드컵로 72,15
1,2025-01-01,103,망원역 2번출구 앞,1,-1.2,0.0,0.7,60,마포구,서울특별시 마포구 월드컵로 79,14
2,2025-01-01,104,합정역 1번출구 앞,1,-1.2,0.0,0.7,60,마포구,서울특별시 마포구 양화로 59,13
3,2025-01-01,105,합정역 5번출구 앞,3,-1.2,0.0,0.7,60,마포구,서울특별시 마포구 양화로 48,5
4,2025-01-01,107,신한은행 서교동지점,1,-1.2,0.0,0.7,60,마포구,서울특별시 마포구 월드컵북로 35,5


In [63]:
# 1. 머지 직후 결측치 현황 확인
print("=== 컬럼별 결측치 개수 ===")
print(df_final_merged.isnull().sum())

=== 컬럼별 결측치 개수 ===
대여일시       0
대여소번호      0
대여소명       0
이용건수       0
기온(°C)     0
강수량(mm)    0
풍속(m/s)    0
습도(%)      0
자치구        0
상세주소       0
거치대수       0
dtype: int64


In [61]:
# [대여소 마스터 정보]
# - 자치구, 상세주소: 신설/임시 대여소라 정보가 누락된 경우 '미확인' 처리
df_final_merged["자치구"] = df_final_merged["자치구"].fillna("미확인")
df_final_merged["상세주소"] = df_final_merged["상세주소"].fillna("미확인")

# - 거치대수: 누락된 경우 전체 대여소 거치대수의 중앙값(Median)으로 채움
df_final_merged["거치대수"] = df_final_merged["거치대수"].fillna(
    df_final_merged["거치대수"].median()
)

In [62]:
print(df_final_merged.isnull().sum().sum())  # 0이 나와야 정상입니다.

0


In [ ]:
df_final_merged.to_csv(
    "bike_final_dataset.csv", index=False, encoding="utf-8-sig"
)

In [1]:
import joblib
import pandas as pd

# 1. 원본 데이터 로드 및 대여소번호 타입 정규화
df = pd.read_csv("../data/bike_final_dataset.csv")

df["대여소번호"] = (
    pd.to_numeric(df["대여소번호"], errors="coerce").fillna(0).astype(int)
)
df["datetime"] = pd.to_datetime(df["대여일시"]).dt.floor("h")

# 2. 대여소 메타데이터 추출 및 [대여소명 가나다순] 정렬
station_info = (
    df.groupby("대여소번호")
    .agg({"대여소명": "first", "자치구": "first", "거치대수": "first"})
    .reset_index()
    .sort_values(by="대여소명")  # 📌 가나다순 정렬 적용
    .reset_index(drop=True)
)

# 3. 시간대별 대여 건수 집계
if "이용건수" in df.columns:
    hourly_df = (
        df.groupby(["대여소번호", "datetime"])["이용건수"]
        .sum()
        .reset_index(name="rental_count")
    )
elif "대여건수" in df.columns:
    hourly_df = (
        df.groupby(["대여소번호", "datetime"])["대여건수"]
        .sum()
        .reset_index(name="rental_count")
    )
else:
    hourly_df = (
        df.groupby(["대여소번호", "datetime"])
        .size()
        .reset_index(name="rental_count")
    )

# 4. [0건 시간대 복원] 전체 대여소 x 전체 시간대 그리드 생성
min_time = hourly_df["datetime"].min()
max_time = hourly_df["datetime"].max()
all_times = pd.date_range(start=min_time, end=max_time, freq="h")

full_grid = (
    pd.MultiIndex.from_product(
        [station_info["대여소번호"].unique(), all_times],
        names=["대여소번호", "datetime"],
    )
    .to_frame()
    .reset_index(drop=True)
)

full_df = pd.merge(
    full_grid,
    hourly_df,
    on=["대여소번호", "datetime"],
    how="left",
)
full_df["rental_count"] = full_df["rental_count"].fillna(0)

# 5. 요일 및 시간대 평균 산출
full_df["dayofweek"] = full_df["datetime"].dt.dayofweek
full_df["hour"] = full_df["datetime"].dt.hour

dow_hour_means = (
    full_df.groupby(["대여소번호", "dayofweek", "hour"])["rental_count"]
    .mean()
    .round(1)
    .reset_index()
)

# 6. 메타데이터 딕셔너리 생성 (가나다순 순서 유지 & 고속화)
# 평균 집계 데이터를 미리 대여소번호별 그룹으로 사전화해 검색 속도 향상
grouped_means = dict(tuple(dow_hour_means.groupby("대여소번호")))

station_meta = {}

# 이미 가나다순으로 정렬된 station_info 순서대로 반복
for _, row in station_info.iterrows():
    s_id = int(row["대여소번호"])
    s_name = str(row["대여소명"])
    district = str(row["자치구"])
    racks = int(row["거치대수"])

    # 해당 대여소의 요일/시간대 평균 딕셔너리 구성
    dow_hour_dict = {}
    if s_id in grouped_means:
        group = grouped_means[s_id]
        dow_hour_dict = {
            f"{int(r['dayofweek'])}_{int(r['hour'])}": float(r["rental_count"])
            for _, r in group.iterrows()
        }

    station_meta[s_id] = {
        "대여소명": s_name,
        "자치구": district,
        "거치대수": racks,
        "station_hour_mean": dow_hour_dict,
    }

joblib.dump(station_meta, "../models/station_metadata.pkl")
print(
    f"✅ pkl 저장 완료! (총 {len(station_meta)}개 대여소)"
)

✅ pkl 저장 완료! (총 2811개 대여소)


In [ ]:
import joblib

station_meta = joblib.load("../models/station_metadata.pkl")

# 1. 데이터 타입 및 첫 번째 항목 구조 확인
sample_key = next(iter(station_meta))
print("데이터 타입:", type(station_meta))
print("내부 샘플 구조:", station_meta[sample_key])

In [3]:
import pandas as pd

# 1. 원본 대여 기록 데이터 로드 (파일명 및 컬럼명은 프로젝트에 맞게 수정)
# 예: df 컬럼 -> ['대여일시', '대여소번호', '자치구', '거치대수']
df = pd.read_csv("data/bike_final_dataset.csv")
df.head(10)

,대여일시,대여소번호,대여소명,이용건수,기온(°C),강수량(mm),풍속(m/s),습도(%),자치구,상세주소,거치대수
0,2025-01-01 00:00:00,102,망원역 1번출구 앞,2,-1.2,0.0,0.7,60,마포구,서울특별시 마포구 월드컵로 72,15.0
1,2025-01-01 00:00:00,103,망원역 2번출구 앞,1,-1.2,0.0,0.7,60,마포구,서울특별시 마포구 월드컵로 79,14.0
2,2025-01-01 00:00:00,104,합정역 1번출구 앞,1,-1.2,0.0,0.7,60,마포구,서울특별시 마포구 양화로 59,13.0
3,2025-01-01 00:00:00,105,합정역 5번출구 앞,3,-1.2,0.0,0.7,60,마포구,서울특별시 마포구 양화로 48,5.0
4,2025-01-01 00:00:00,107,신한은행 서교동지점,1,-1.2,0.0,0.7,60,마포구,서울특별시 마포구 월드컵북로 35,5.0
5,2025-01-01 00:00:00,108,서교동 사거리,1,-1.2,0.0,0.7,60,마포구,서울특별시 마포구 양화로 93,10.0
6,2025-01-01 00:00:00,111,상수역 2번출구 앞,1,-1.2,0.0,0.7,60,마포구,서울특별시 마포구 와우산로 40,10.0
7,2025-01-01 00:00:00,113,홍대입구역 2번출구 앞,3,-1.2,0.0,0.7,60,마포구,서울특별시 마포구 양화로 165,25.0
8,2025-01-01 00:00:00,114,홍대입구역 8번출구 앞,2,-1.2,0.0,0.7,60,마포구,서울특별시 마포구 양화로18길 3,15.0
9,2025-01-01 00:00:00,120,신수동 사거리,1,-1.2,0.0,0.7,60,마포구,서울특별시 마포구 토정로 211,5.0
